# CRISPR-Cas13 Diagnostic Guide RNA Design Tool

An automated pipeline for designing CRISPR-Cas13 guide RNAs (crRNAs) that target conserved pathogen genomic regions for nucleic acid-based diagnostics.

## Background

CRISPR-Cas13 systems have emerged as powerful platforms for nucleic acid detection, underpinning diagnostic technologies such as **SHERLOCK** (Specific High-sensitivity Enzymatic Reporter unLOCKing). Unlike Cas9, Cas13 targets **RNA** and exhibits collateral cleavage activity upon target recognition, enabling signal amplification for attomolar-level sensitivity.

The efficacy of a Cas13-based diagnostic depends critically on **guide RNA design**:
- The guide must target a **highly conserved** region to detect diverse pathogen strains
- It must avoid **cross-reactivity** with the human transcriptome
- Biophysical properties (GC content, secondary structure, homopolymers) affect enzymatic activity

This notebook implements a systematic, multi-criteria pipeline for guide RNA selection.

## Pipeline Overview

```
Pathogen Input (name / taxonomy ID)
        │
        ▼
1. Sequence Retrieval (NCBI Entrez)
        │
        ▼
2. Multiple Sequence Alignment (MUSCLE)
        │
        ▼
3. Conserved Region Identification
        │
        ▼
4. Guide RNA Candidate Extraction & Filtering
        │
        ▼
5. Off-Target Screening (BLAST vs. human transcriptome)
        │
        ▼
6. Multi-Criteria Scoring & Ranking
        │
        ▼
7. Visualization & Results
```

---
## 1. Setup & Configuration

### Dependencies
- **BioPython** — NCBI Entrez queries, BLAST, sequence I/O
- **MUSCLE** — Multiple sequence alignment (must be installed and on PATH)
- **pandas / numpy** — Data manipulation and scoring
- **matplotlib / seaborn** — Visualization

In [ ]:
import sys
sys.path.insert(0, "..")

from cas13_design.retrieval import fetch_sequences
from cas13_design.alignment import run_alignment, compute_conservation
from cas13_design.guide_design import extract_candidates
from cas13_design.off_target import screen_off_targets
from cas13_design.scoring import score_guides, rank_guides
from cas13_design.visualization import (
    plot_conservation_heatmap,
    plot_score_breakdown,
    plot_gc_distribution,
    plot_conservation_vs_offtarget,
)

import warnings
warnings.filterwarnings("ignore")

print("All modules loaded successfully.")

### Configuration

Specify the target pathogen and optional gene. The pipeline will query NCBI for available sequences.

In [ ]:
# === USER CONFIGURATION ===
PATHOGEN = "Zika virus"         # Pathogen name or NCBI taxonomy ID
GENE_TARGET = None               # Optional: e.g., "NS5", "E", "RdRp" (None = whole genome)
MAX_SEQUENCES = 50               # Number of sequences to retrieve from NCBI

# Guide RNA parameters
GUIDE_LENGTH = 28                # Cas13a optimal guide length (nt)
CONSERVATION_THRESHOLD = 0.90    # Minimum conservation to consider a region
GC_MIN = 0.40                    # Minimum GC content
GC_MAX = 0.60                    # Maximum GC content
MAX_HOMOPOLYMER = 4              # Maximum allowed homopolymer run

# Off-target screening
BLAST_EVALUE = 1e-5              # E-value threshold for human BLAST
MAX_GUIDES_TO_BLAST = 20         # Limit BLAST queries (rate-limited)

print(f"Target: {PATHOGEN}")
print(f"Gene: {GENE_TARGET or 'All (genome-wide)'}")
print(f"Guide length: {GUIDE_LENGTH} nt")

---
## 2. Sequence Retrieval

We query NCBI's nucleotide database using Entrez to retrieve pathogen sequences. The search filters for sequences between 200–10,000 bp to capture gene-level or partial genome sequences suitable for alignment.

In [ ]:
sequences = fetch_sequences(
    pathogen=PATHOGEN,
    gene=GENE_TARGET,
    max_seqs=MAX_SEQUENCES,
)

# Summary statistics
lengths = [len(r.seq) for r in sequences]
print(f"\nSequence length range: {min(lengths)}–{max(lengths)} bp")
print(f"Mean length: {sum(lengths) / len(lengths):.0f} bp")

---
## 3. Multiple Sequence Alignment

MUSCLE aligns the retrieved sequences to identify positional homology. This is essential for computing conservation — positions that are invariant across diverse isolates are ideal diagnostic targets, as they are less likely to be lost through viral evolution.

In [ ]:
alignment = run_alignment(sequences)

### Conservation Analysis

We compute a per-position conservation score: the fraction of sequences sharing the most common nucleotide at each column. A sliding window of 28 nt (matching guide length) then averages these scores to identify contiguous conserved regions.

In [ ]:
window_scores, position_scores = compute_conservation(
    alignment, window_size=GUIDE_LENGTH
)

import numpy as np
print(f"Alignment length: {len(position_scores)} positions")
print(f"Mean conservation: {np.mean(position_scores):.3f}")
print(f"Positions with >90% conservation: {(position_scores > 0.90).sum()}")
print(f"28-nt windows with >90% mean conservation: {(window_scores > 0.90).sum()}")

In [ ]:
fig = plot_conservation_heatmap(position_scores)
fig.savefig("conservation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 4. Guide RNA Candidate Extraction

From the conserved regions, we extract all possible 28-nt guide RNA sequences and apply biophysical filters:

| Filter | Criterion | Rationale |
|--------|-----------|----------|
| Conservation | ≥90% mean across window | Ensures broad strain coverage |
| GC content | 40–60% | Optimal for Cas13 activity and probe stability |
| Homopolymer | Max run ≤4 nt | Long runs reduce cleavage efficiency |
| Self-complementarity | Low score preferred | Guides that self-fold have reduced target binding |

In [ ]:
candidates = extract_candidates(
    alignment=alignment,
    window_scores=window_scores,
    guide_length=GUIDE_LENGTH,
    conservation_threshold=CONSERVATION_THRESHOLD,
    gc_min=GC_MIN,
    gc_max=GC_MAX,
    max_homopolymer=MAX_HOMOPOLYMER,
)

if not candidates.empty:
    display(candidates.head(10))

In [ ]:
import matplotlib.pyplot as plt

fig = plot_gc_distribution(candidates)
fig.savefig("gc_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 5. Off-Target Screening

Each candidate guide is BLASTed against the **human RefSeq transcriptome** to check for potential cross-reactivity. A diagnostic guide that triggers on human RNA would produce false positives.

We flag any guide with an alignment of ≥80% identity over ≥20 nt against a human transcript.

> **Note:** This step queries NCBI's BLAST servers and may take several minutes due to rate limiting (~3 s between queries).

In [ ]:
screened = screen_off_targets(
    candidates=candidates,
    evalue=BLAST_EVALUE,
    delay=3.0,
    max_guides=MAX_GUIDES_TO_BLAST,
)

---
## 6. Multi-Criteria Scoring & Ranking

Guides are scored using a weighted composite of five criteria:

| Criterion | Weight | Direction |
|-----------|--------|-----------|
| Conservation | 30% | Higher is better |
| Off-target safety | 25% | Fewer human hits is better |
| GC content optimality | 15% | Closer to 50% is better |
| Homopolymer penalty | 15% | Shorter max run is better |
| Self-complementarity | 15% | Lower self-folding is better |

The composite score ranges from 0 to 1, where 1 represents an ideal guide.

In [ ]:
scored = score_guides(screened)
ranked = rank_guides(scored, top_n=10)

print("\n=== TOP 10 GUIDE RNA CANDIDATES ===")
display(ranked)

---
## 7. Visualization

### Conservation Heatmap with Candidate Positions

Blue dashed lines indicate the positions of guide RNA candidates overlaid on the genome-wide conservation profile.

In [ ]:
fig = plot_conservation_heatmap(position_scores, candidates=screened)
fig.savefig("conservation_with_guides.png", dpi=150, bbox_inches="tight")
plt.show()

### Score Breakdown

Stacked bar chart showing how each scoring criterion contributes to the composite score for the top-ranked guides.

In [ ]:
fig = plot_score_breakdown(ranked)
fig.savefig("score_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

### Conservation vs. Off-Target Specificity

Ideal guides occupy the **upper-right** region: high conservation (broad strain coverage) with zero off-target human hits (no false positives).

In [ ]:
fig = plot_conservation_vs_offtarget(scored)
fig.savefig("conservation_vs_offtarget.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 8. Results Summary

### Top 5 Recommended Guide RNAs

In [ ]:
top5 = ranked.head(5)

print("=" * 80)
print(f"CRISPR-Cas13 Guide RNA Design Report: {PATHOGEN}")
print(f"Gene target: {GENE_TARGET or 'Genome-wide'}")
print(f"Sequences analysed: {len(sequences)}")
print(f"Candidates after filtering: {len(candidates)}")
print(f"Guides screened for off-targets: {len(screened)}")
print("=" * 80)

for _, row in top5.iterrows():
    print(f"\n--- Guide #{int(row['rank'])} ---")
    print(f"  Sequence (5'→3'):  {row['sequence']}")
    print(f"  Alignment position: {int(row['position'])}")
    print(f"  Composite score:    {row['composite_score']:.4f}")
    print(f"  Conservation:       {row['conservation']:.4f}")
    print(f"  GC content:         {row['gc_content']:.1%}")
    print(f"  Off-target hits:    {int(row['off_target_hits'])}")

### Export Results

In [ ]:
output_file = f"{PATHOGEN.replace(' ', '_')}_cas13_guides.csv"
ranked.to_csv(output_file, index=False)
print(f"Results exported to {output_file}")

---

## Methods Summary

This pipeline integrates:
1. **NCBI Entrez** for automated pathogen sequence retrieval across diverse isolates
2. **MUSCLE** multiple sequence alignment for identifying positionally conserved genomic regions
3. **Biophysical filtering** (GC content, homopolymer runs, self-complementarity) to select guides with favourable enzymatic properties
4. **NCBI BLAST** off-target screening against the human RefSeq transcriptome to ensure diagnostic specificity
5. **Weighted multi-criteria scoring** balancing sensitivity (conservation) and specificity (off-target safety) with biophysical optimality

The modular design allows extension to additional scoring criteria, alternative alignment tools, or local BLAST databases.